In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.gold;

In [0]:
%sql
DROP TABLE IF EXISTS workspace.gold.dim_territorio;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.gold.dim_territorio (
    id_ubigeo STRING COMMENT 'Código único de distrito según INEI',
    superficie_km2 DOUBLE COMMENT 'Superficie del distrito en km²',
    altitud DOUBLE COMMENT 'Altitud promedio del distrito en metros sobre el nivel del mar',
    categoria STRING COMMENT 'Clasificación del distrito (Ciudad, Villa, Pueblo)',
    capital_legal STRING COMMENT 'Capital legal del distrito',
    fecha_actualizacion DATE COMMENT 'Fecha de actualización de la tabla Gold'
) USING DELTA;

In [0]:
df_ubi_equi = spark.table("workspace.silver.m_ubigeo_equivalencia").alias("A")
df_ubi_geo = spark.table("workspace.silver.m_ubigeo_geografia").alias("B")

df_ubi_geo_homologado =  df_ubi_equi.join(df_ubi_geo, "id_ubigeo_reniec", "left").select("A.id_ubigeo", "B.superficie").alias("C")

df_ubi_demo = spark.table("silver.m_ubigeo_demografia").alias("D")

dim_territorio = (df_ubi_demo
    .join(df_ubi_geo_homologado, "id_ubigeo", "left")
    .select(
        "D.id_ubigeo",
        col("C.superficie").alias("superficie_km2"),
        "D.altitud",
        "D.categoria",
        "D.capital_legal"
    )
    .withColumn("fecha_actualizacion", current_date())
)

dim_territorio.write.format("delta").mode("overwrite").saveAsTable("workspace.gold.dim_territorio")